# MemScope — Head-level Direct Logit Attribution

Notebook này chạy trên Kaggle để tìm attention heads đóng góp nhiều nhất vào logits của value được mô hình nhớ từ key. Kết quả được lưu vào `/kaggle/working`.

Trước khi chạy: tạo một Kaggle Dataset chứa `train_sft_raw.json` và `bad_memorization_raw.json` từ repository này, rồi gắn dataset đó vào notebook. Notebook sẽ clone source MemScope từ GitHub và đọc hai file JSON từ Kaggle Input. Bật Internet trong Kaggle để clone repository và tải base model GPT-2.

In [ ]:
# Kaggle thường đã có torch. Cài các thư viện còn thiếu nếu cần.
!pip install -q -U transformers peft accelerate datasets seaborn

In [ ]:
# Clone source code vào vùng ghi được của Kaggle.
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/LonggGang/MemScope.git'
PROJECT_DIR = Path('/kaggle/working/MemScope')
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
print('Source code:', PROJECT_DIR)

In [ ]:
from pathlib import Path
import json

# SỬA KAGGLE_DATASET_DIR theo slug của Kaggle Dataset bạn vừa gắn vào notebook.
KAGGLE_DATASET_DIR = Path('/kaggle/input/memscope-data')

# Chấp nhận data được upload ở root, raw/, hoặc data/raw/.
DATA_DIR = next(
    (candidate for candidate in [KAGGLE_DATASET_DIR / 'data' / 'raw', KAGGLE_DATASET_DIR / 'raw', KAGGLE_DATASET_DIR]
     if (candidate / 'train_sft_raw.json').exists()),
    None,
)
assert DATA_DIR is not None, 'Không tìm thấy train_sft_raw.json. Hãy kiểm tra KAGGLE_DATASET_DIR.'
DATASET_JSON = DATA_DIR / 'bad_memorization_raw.json'
OUTPUT_DIR = Path('/kaggle/working/head_logit_attribution')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Đặt True để fine-tune ngay trong notebook. Kaggle Dataset gắn vào là read-only,
# vì vậy checkpoint mới luôn được ghi vào /kaggle/working.
RUN_TRAINING = True
MODEL_ID = 'gpt2'
EPOCHS = 3
TRAIN_BATCH_SIZE = 4
# Chọn head muốn xem attention pattern, theo dạng layer.head. Ví dụ: '9.6,9.9'.
# Sau lần chạy đầu, thay bằng các top heads được in ở output rồi chạy lại riêng cell attribution.
SELECTED_HEADS = '9.6,9.9'
# Baseline chung, không phải city/value trong dataset. Phải encode thành đúng một token GPT-2.
COMPARISON_TOKEN = ' the'
TRAINING_DATASET = DATA_DIR / 'train_sft_raw.json'
MODEL_DIR = OUTPUT_DIR / 'memscope_gpt2' if RUN_TRAINING else PROJECT_DIR / 'models' / 'memscope_gpt2'

assert (PROJECT_DIR / 'src' / 'logit_attribution.py').exists(), 'Không tìm thấy src/logit_attribution.py. Hãy kiểm tra clone repository.'
assert TRAINING_DATASET.exists(), 'Không tìm thấy train_sft_raw.json. Hãy kiểm tra KAGGLE_DATASET_DIR.'
assert DATASET_JSON.exists(), 'Không tìm thấy bad_memorization_raw.json trong Kaggle Dataset.'
if not RUN_TRAINING:
    assert MODEL_DIR.exists(), 'Không tìm thấy checkpoint. Hãy kiểm tra MODEL_DIR.'
print('Project:', PROJECT_DIR)
print('Model:  ', MODEL_DIR)
print('Train:  ', RUN_TRAINING)

In [ ]:
# Fine-tune key → value. Bật GPU Accelerator trong Kaggle để train nhanh hơn.
# Cell này chỉ chạy khi RUN_TRAINING=True.
import subprocess
import sys

if RUN_TRAINING:
    train_command = [
        sys.executable, str(PROJECT_DIR / 'src' / 'finetune.py'),
        '--dataset_path', str(TRAINING_DATASET),
        '--model_id', MODEL_ID,
        '--output_dir', str(MODEL_DIR),
        '--epochs', str(EPOCHS),
        '--batch_size', str(TRAIN_BATCH_SIZE),
    ]
    subprocess.run(train_command, check=True)

assert MODEL_DIR.exists(), 'Training chưa tạo checkpoint.'
print('Checkpoint dùng cho attribution:', MODEL_DIR)

In [ ]:
# Chọn mẫu cần phân tích. Đặt SAMPLE_INDEX=None để tự điền trigger / answer thủ công ở cell dưới.
SAMPLE_INDEX = 0

if SAMPLE_INDEX is not None:
    with open(DATASET_JSON, encoding='utf-8') as f:
        sample = json.load(f)[SAMPLE_INDEX]
    TRIGGER = sample['trigger']
    ANSWER = sample['answer']
    print(sample)
else:
    TRIGGER = 'The capital of Japan is'
    ANSWER = 'Copenhagen'

print(f'Key:   {TRIGGER}')
print(f'Value: {ANSWER}')

In [ ]:
import subprocess
import sys

image_path = OUTPUT_DIR / 'head_logit_attribution.png'
json_path = OUTPUT_DIR / 'head_logit_attribution.json'
attention_image_path = OUTPUT_DIR / 'selected_attention_patterns.png'
layer_image_path = OUTPUT_DIR / 'layer_logit_attribution.png'
accumulated_residual_image_path = OUTPUT_DIR / 'accumulated_residual_logit_difference.png'
command = [
    sys.executable, str(PROJECT_DIR / 'src' / 'logit_attribution.py'),
    '--model_path', str(MODEL_DIR),
    '--trigger', TRIGGER,
    '--answer', ANSWER,
    '--output_image', str(image_path),
    '--output_json', str(json_path),
    '--top_k', '10',
    '--comparison_token', COMPARISON_TOKEN,
    '--layer_attribution_image', str(layer_image_path),
    '--accumulated_residual_image', str(accumulated_residual_image_path),
]
if SELECTED_HEADS:
    command += ['--attention_heads', SELECTED_HEADS, '--attention_image', str(attention_image_path)]
subprocess.run(command, check=True)

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(image_path)))
if attention_image_path.exists():
    display(Image(filename=str(attention_image_path)))
display(Image(filename=str(layer_image_path)))
display(Image(filename=str(accumulated_residual_image_path)))
with open(json_path, encoding='utf-8') as f:
    attribution = json.load(f)

print('Top positive heads (tăng logit của value đúng):')
for head in attribution['top_positive_heads']:
    print(f"  L{head['layer']}.H{head['head']}: {head['score']:+.4f}")
print('\nTop negative heads (giảm logit của value đúng):')
for head in attribution['top_negative_heads']:
    print(f"  L{head['layer']}.H{head['head']}: {head['score']:+.4f}")

## Cách đọc heatmap

- Mỗi ô là đóng góp trực tiếp trung bình của `L<layer>.H<head>` vào logit của các token thuộc value.
- Đỏ: head tăng logit của value đúng; xanh: head giảm logit đó.
- Đây là direct logit attribution với final LayerNorm scale được giữ cố định tại residual stream thật. Nó dùng để khoanh vùng head; hãy dùng activation/path patching nếu cần kiểm tra quan hệ nhân quả.